# 06 · Data splits, window pool and teacher

The experiment of the paper runs on PyNAS 1.0 from start to end (`pyqnas/docs/DESIGN.md`):

1. **teacher**: a ResNet-50 U-Net trained on the training split; its class probabilities are the soft labels;
2. **search**: NSGA-II, 50 candidates per generation, 3 generations; every candidate checked for the Myriad X before training, trained in FP16 with distillation, BatchNorm folded for the last 20% of its steps, scored in FP16 on the search split;
3. **final**: the winner trained again with 3 seeds, baselines and ablations through the same trainer, the deployed model exported for ArgOS, the test split read once.

This notebook makes the data the rest stands on, and the teacher. Splits are by whole scenes or groups of scenes (a group: the same pass, or overlapping ground):

| split | used for | sets in the pool |
|---|---|---|
| train | teacher and candidates learn | `train` |
| search | fitness of the candidates | `heo_search` |
| val | choice of the deployed seed | `heo_val`, `phisat2_val` |
| test | the final numbers, read once | `heo_test`, `phisat2_test`, `s2_test` |

The search split used during development was a single HEO scene: differences between candidates were of the size of the noise of one scene. Here it is several HEO groups, about a fifth of the HEO training fresh burn, held out of training.

In [ ]:
# [1] setup
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                # the notebooks run from the repository root
import os, sys, json, time, shutil
from itertools import combinations
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
for p in (os.path.join(os.getcwd(), "pyqnas", "src"), os.getcwd()):
    if p not in sys.path:
        sys.path.insert(0, p)
import pynas
from pynas import config, runner, teacher
import burnscar.labels as L
import burnscar.dataset as D
import burnscar.pool as P

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 80)
C_BLUE, C_ORANGE, C_GREY, INK2 = "#2a78d6", "#eb6834", "#8a8984", "#52514e"
plt.rcParams.update({"axes.grid": True, "grid.color": "#e6e5e0", "axes.axisbelow": True, "axes.spines.top": False,
                     "axes.spines.right": False, "legend.frameon": False, "axes.edgecolor": C_GREY,
                     "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2})
assert pynas.__version__.startswith("1."), f"PyNAS {pynas.__version__}: PyNAS 1.0 is not in place"
dcfg = json.load(open("data/tables/dataset_config.json"))
scenes = pd.read_csv(dcfg["files"]["scenes"])
norm = json.load(open(dcfg["files"]["normalisation"]))
SIZE = dcfg["size"]
POOL = "data/pool_v2"
CONFIG = Path("configs/burnscar.ini")
CORES = len(os.sched_getaffinity(0))
print(f"PyNAS {pynas.__version__}; {len(scenes)} scenes; windows of {SIZE} px; {CORES} CPU cores; "
      f"{shutil.disk_usage('data').free / 1e9:.0f} GB free on the data disk")

## 1. The experiment's configuration

One file, `configs/burnscar.ini`, holds every choice of the experiment; PyNAS takes its defaults for the rest and writes the effective configuration into every run folder. It is made here once, from the dataset configuration of 04 (class names and weights, source shares), and kept in the repository afterwards; the cell shows it when it exists. `ARGOS_LAYERS` is the ArgOS test-suite list of Myriad layers (`myriad_compatible_layers.txt` in the SDK): IR layer types missing from it make a candidate infeasible.

In [ ]:
# [2] configs/burnscar.ini
ARGOS_LAYERS = ""   # Argotec's list of supported layer types is not public; empty: every candidate is still checked to compile for MYRIAD
if not CONFIG.exists():
    CONFIG.parent.mkdir(parents=True, exist_ok=True)
    shares = ", ".join(f"{k}: {v}" for k, v in dcfg["source_weights"].items())
    CONFIG.write_text(f"""; The burn-scar experiment of the paper. PyNAS defaults (pyqnas/src/pynas/config.ini) apply to every option
; not set here.

[Data]
pool = {POOL}
train = train
search = heo_search
val = heo_val, phisat2_val
test = heo_test, phisat2_test, s2_test
norm = {dcfg['files']['normalisation']}
class_names = {', '.join(dcfg['class_names'])}
class_weights = {', '.join(str(w) for w in dcfg['class_weights'])}
target = fresh
group = burn: fresh, old
source_share = {shares}
teacher = teacher

[Hardware]
argos_layers = {ARGOS_LAYERS}

[Run]
out = models/pynas
""")
    print(f"written: {CONFIG}")
cfg = config.load(CONFIG)
print(CONFIG.read_text())

## 2. The search split

Whole HEO training groups, L1C, geolocation within 8 m, at least 3 groups, together as close as possible to 20% of the HEO training fresh burn; the group with the most fresh burn stays in training. To choose by hand, list scene names (or their start) in `SEARCH_SCENES`: a scene's whole group goes with it.

In [ ]:
# [3] HEO training groups and the ones held out for the search
TARGET, MIN_GROUPS, MAX_GEO = 0.20, 3, 8.0
SEARCH_SCENES = None
heo_tr = scenes[(scenes.source == "heo") & (scenes.split == "train")]
g = heo_tr.groupby("group").agg(scenes=("scene", "size"), first_scene=("scene", "min"), fresh_px=("fresh_px", "sum"),
                                 old_px=("old_px", "sum"), levels=("level", lambda s: ",".join(sorted(set(s)))),
                                 geo_m=("geo_rms_m", "max"))
total = max(int(g.fresh_px.sum()), 1)
g["fresh_%"] = (100 * g.fresh_px / total).round(1)
g = g.sort_values("fresh_px", ascending=False)
if SEARCH_SCENES is None:
    usable = (g.fresh_px > 0) & (g.levels == "L1C") & (g.index != g.fresh_px.idxmax())
    cand = g[usable & (g.geo_m <= MAX_GEO)]
    if len(cand) < MIN_GROUPS:                 # too few with a known geolocation: allow those without metadata
        cand = g[usable & ~(g.geo_m > MAX_GEO)]
    combos = [list(c) for k in range(MIN_GROUPS, min(MIN_GROUPS + 3, len(cand)) + 1) for c in combinations(cand.index, k)]
    assert combos, f"fewer than {MIN_GROUPS} usable HEO groups: set SEARCH_SCENES by hand"
    pick = min(combos, key=lambda c: abs(g.loc[c, "fresh_px"].sum() / total - TARGET))
    search = heo_tr[heo_tr.group.isin(pick)]
else:
    named = heo_tr[heo_tr.scene.str.startswith(tuple(SEARCH_SCENES))]
    search = heo_tr[heo_tr.group.isin(named.group)]
g["search"] = g.index.isin(search.group)
display(g.drop(columns=["first_scene"]))
share = search.fresh_px.sum() / total
print(f"search split: {len(search)} scenes in {search.group.nunique()} groups, {int(search.fresh_px.sum()):,} fresh pixels "
      f"({100 * share:.0f}% of the HEO training fresh burn); training keeps {len(heo_tr) - len(search)} HEO scenes")
for s in search.scene:
    print("   ", s)

## 3. The window pool

Training windows drawn with the settings of 04 (sources, shares of windows centred on fresh and old burn, resolutions), as many as 30% of the free disk allows (at most 24,000); the search scenes whole at 2.5, 5 and 10 m; the validation and test windows of 04 (PhiSat-2: at most 1,000 windows with fresh burn and 500 without). Written once into `data/pool_v2/`. All cores, about an hour. The table of [5] ends with the scenes and the groups of each scoring set that are also in the training windows: no scene may be (the cell stops), and a shared group (overlapping ground) is reported as it is.

In [ ]:
# [4] the training windows
os.makedirs(POOL, exist_ok=True)
points = D.load_points(dcfg["files"]["points"])
free = shutil.disk_usage(POOL).free
per = P.bytes_per_window(SIZE)
N = int(min(24000, 0.3 * free / per) // 1000 * 1000)
assert N >= 2000, f"only room for {N} windows: free some disk first"
if not Path(f"{POOL}/train_windows.csv").exists():
    pool_scenes = scenes[(scenes.split == "train") & ~scenes.scene.isin(search.scene)]
    ds = D.TrainWindows(pool_scenes, points, norm, size=SIZE, length=N, source_weights=dcfg["source_weights"],
                        gsd_range=dcfg["gsd_range"], p_fresh=dcfg["p_fresh"], p_old=dcfg["p_old"],
                        valid_scenes=dcfg.get("valid_scenes"), augment=False, seed=100)
    t0 = time.time()
    P.write_set(ds, f"{POOL}/train", workers=max(1, CORES - 1))
    print(f"{N:,} training windows written in {(time.time() - t0) / 60:.1f} min")
tw = pd.read_csv(f"{POOL}/train_windows.csv")
assert not set(tw.scene) & set(search.scene), "a search scene is in the training windows"
tw.groupby(["source", "kind"]).size().unstack(fill_value=0)

In [ ]:
# [5] the scoring sets: search, validation, test
wv, wt = pd.read_csv(dcfg["files"]["windows_val"]), pd.read_csv(dcfg["files"]["windows_test"])


def sample_phisat2(w):
    return pd.concat([w[w.fresh > 0].sample(min(1000, int((w.fresh > 0).sum())), random_state=0),
                      w[w.fresh == 0].sample(min(500, int((w.fresh == 0).sum())), random_state=0)])


SETS = {"heo_search": P.grid_windows(search, dcfg["eval_gsd"]["heo"], SIZE),
        "heo_val": wv[wv.source == "heo"], "phisat2_val": sample_phisat2(wv[wv.source == "phisat2_sim"]),
        "heo_test": wt[wt.source == "heo"], "phisat2_test": sample_phisat2(wt[wt.source == "phisat2_sim"]),
        "s2_test": wt[~wt.source.isin(["heo", "phisat2_sim"])]}
train_groups = set(scenes[scenes.scene.isin(tw.scene)].group)
rows = []
for name, w in SETS.items():
    if len(w) and not Path(f"{POOL}/{name}_windows.csv").exists():
        w = w.reset_index(drop=True)
        P.write_set(D.FixedWindows(w, scenes, norm, size=SIZE), f"{POOL}/{name}", workers=max(1, CORES - 1), every=0,
                    extra={"fresh": w.fresh.values, "old": w.old.values})
    if Path(f"{POOL}/{name}_windows.csv").exists():
        t = pd.read_csv(f"{POOL}/{name}_windows.csv")
        rows.append(dict(set=name, windows=len(t), scenes=t.scene.nunique(), with_fresh=int((t.fresh > 0).sum()),
                         gsd=sorted(t.gsd.round(2).unique().tolist()),
                         in_training=len(set(t.scene) & set(tw.scene)),
                         groups_shared_with_training=len(set(scenes[scenes.scene.isin(t.scene)].group) & train_groups)))
sets_table = pd.DataFrame(rows).set_index("set")
assert (sets_table.in_training == 0).all(), "a scoring scene is also in the training windows"
sets_table

## 4. The teacher

`[Teacher]` of the configuration (PyNAS defaults): smp U-Net with a ResNet-50 encoder pretrained on ImageNet, 7 input bands, 20,000 batches of 16, AdamW at 3e-4 with 500 warm-up steps and a cosine decay, cross-entropy with the class weights plus soft Dice, bfloat16 mixed precision, BatchNorm frozen and the weights averaged (EMA) over the last 20% of the steps; seeds 0 and 1, one per GPU, in the background. It is trained by the same trainer as every candidate and scored on the search and validation sets in FP32.

In [ ]:
# [6] the teacher, 2 seeds in the background (running the cell again carries on)
GPUS = list(range(torch.cuda.device_count())) or [0]
TQ = teacher.queue(cfg, Path(cfg.get("Run", "out")) / "teacher")
runner.launch(TQ, gpus=GPUS)

In [ ]:
# [7] progress every 10 minutes (interrupting stops the watching, not the training)
runner.wait(TQ, every=600)

In [ ]:
# [8] the teacher's scores (fresh-burn IoU, FP32), then its soft labels written into the pool
Q = runner.read_queue(TQ)
spec_sets = config.get_list(cfg, "Data", "search") + config.get_list(cfg, "Data", "val")
rows = []
for r in Q["runs"]:
    sc = runner.done_scores(Q, r)
    if sc:
        rows.append(dict(seed=r["seed"], run=r["run"], minutes=round(sc["minutes"]),
                         **{n: sc.get(f"{n}_fp32_labels_fresh_iou") for n in spec_sets}))
display(pd.DataFrame(rows).set_index("seed").round(3))
if len(rows) == len(Q["runs"]):
    t0 = time.time()
    written = teacher.write_soft_labels(TQ, cfg)
    print(f"soft labels (mean of the {len(rows)} seeds, 8 rotations and flips) written in {(time.time() - t0) / 60:.0f} min:")
    for n, p in written.items():
        print(f"   {p}")

Next: `07_search.ipynb`, the search.